# Resell Copilot — Qwen v3 field-eval

Re-runs the field-level evaluation of `mchlkan/qwen3vl4b-resell-adapter-multi-v3` on the locked 500-row test split.
Results land in `eval/results/qwen_field_eval.json` and overwrite the stale v1 numbers from 2026-05-08.

**Before running:** Runtime → Change runtime type → **GPU** (free T4 is enough with `--load-in-4bit`).

Total time: ~30 min (T4) or ~15 min (A100/L4). The script checkpoints every 25 rows, so a disconnected runtime can be resumed by re-running the same cell.

### If anything breaks on Colab

Colab caches `pip install` results across restarts. A normal **Restart runtime** does **not** undo a broken install. If you hit `BloomPreTrainedModel`-style import errors after `pip install`, do **Runtime → Disconnect and delete runtime**, reconnect to a fresh GPU runtime, and start from cell 1.


In [ ]:
# 1. GPU check
!nvidia-smi


In [ ]:
# 2. Clone the repo (skip the clone if Advanced_ML/ already exists, just pull)
import os
if not os.path.isdir('Advanced_ML'):
    !git clone https://github.com/mchlkan/Advanced_ML.git
%cd Advanced_ML
!git pull --ff-only origin main


In [ ]:
# 3. Install dependencies — DO NOT upgrade torch (Colab's pre-installed torch is fine
#    and upgrading it can break torchvision, which cascades into transformers'
#    image_utils failing to import).
#    Transformers floor is 4.56 — that's where Qwen3-VL support landed; older versions
#    will fail with `KeyError: 'qwen3_vl'` when loading the model config.
!pip install -q "transformers>=4.56,<5" peft accelerate bitsandbytes \
               pandas pyarrow pillow tqdm


In [ ]:
# 4. Sanity-check the install BEFORE attempting the smoke test.
# If this prints 'all good', dependencies are wired up correctly.
# If it errors, do Runtime → Disconnect and delete runtime, then start over from cell 1.
%%bash
python - <<'PY'
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel
import bitsandbytes, accelerate, torch, transformers
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('all good')
PY


## Data files

The script reads two big parquets from `data/`:

- `vinted_clothing_combined.parquet`
- `kleinanzeigen_clothing_combined.parquet`

Both are gitignored, so you need to ship them into the Colab session. The locked **split manifests** (`data/splits/test_ids.json`, etc.) are already in the cloned repo — no need to copy those.

**Pick the option that matches where you keep the parquets** (only run one of the next two cells).


In [ ]:
# Option A — pull the two combined parquets from Google Drive
# (the split MANIFESTS are already in the repo at data/splits/*.json — no need to copy them)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SRC = '/content/drive/MyDrive/Resell_Copilot_data'   # ← edit this if your folder is named differently

os.makedirs('data', exist_ok=True)
for fn in ['vinted_clothing_combined.parquet', 'kleinanzeigen_clothing_combined.parquet']:
    src_path = f'{SRC}/{fn}'
    if not os.path.exists(src_path):
        raise FileNotFoundError(f'expected {src_path} on Drive — adjust SRC above to where you keep it')
    shutil.copy(src_path, f'data/{fn}')

!ls -la data/*.parquet data/splits/


In [ ]:
# Option B — upload manually via the Files panel on the left
# Drag the two parquet files into data/ in the Colab file browser, then verify here
!ls -la data/*.parquet 2>/dev/null && echo '--- splits already in repo:' && ls -la data/splits/


In [ ]:
# 5. Sanity check — confirm the script's expected inputs are in place
import pathlib
required = [
    'data/vinted_clothing_combined.parquet',
    'data/kleinanzeigen_clothing_combined.parquet',
    'data/splits/test_ids.json',
]
missing = [p for p in required if not pathlib.Path(p).exists()]
for p in required:
    print(f'{p:60s}  {"OK" if pathlib.Path(p).exists() else "MISSING"}')
if missing:
    raise SystemExit(f'\nMissing files: {missing}. Re-run the data-copy cell.')


## (Optional) HuggingFace token

The v3 adapter is public, so `HF_TOKEN` is only needed if you hit a rate limit during the base-model download. Uncomment and set if needed.


In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'


## Smoke test (5 rows, ~2 min)

If this succeeds, the full benchmark will too. Confirms the script imports cleanly, the model loads on this GPU, and the prompt schema parses end-to-end.


In [ ]:
!python eval/run_qwen_field_eval.py --load-in-4bit --limit 5 \
  --predictions /tmp/qwen_pred_smoke.parquet \
  --metrics /tmp/qwen_eval_smoke.json

import json, os
if os.path.exists('/tmp/qwen_eval_smoke.json'):
    with open('/tmp/qwen_eval_smoke.json') as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print('Smoke run failed — see the script output above for the actual error')


## Full 500-row benchmark (~20–30 min on T4, ~10–15 min on A100/L4)

Output overwrites `eval/results/qwen_field_eval.json` and `eval/results/qwen_field_predictions.parquet`. If the runtime disconnects, just re-run this cell — the per-row predictions parquet is checkpointed every 25 rows and will resume.


In [ ]:
!python eval/run_qwen_field_eval.py --load-in-4bit


In [ ]:
# 6. Inspect the result file and pull out the headline numbers for the slide
import json
with open('eval/results/qwen_field_eval.json') as f:
    metrics = json.load(f)

print('=== full JSON ===')
print(json.dumps(metrics, indent=2))

print('\n=== headline numbers ===')
print(f'  adapter:           {metrics.get("adapter_id")}')
print(f'  total rows:        {metrics.get("total")}')
if metrics.get('parse_rate') is not None:
    print(f'  parse_rate:        {metrics["parse_rate"]:.1%}   (v1: 81.8%)')

acc = metrics.get('accuracy', {})
v1_baseline = {'brand': 0.475, 'category': 0.968, 'condition': 0.656, 'color': 0.722, 'size': 0.240}
for field in ('brand', 'category', 'condition', 'color', 'size'):
    a = acc.get(field, {})
    if a.get('exact') is not None:
        v1 = v1_baseline.get(field)
        delta = f'  (Δ {(a["exact"] - v1) * 100:+.1f} pp vs v1 {v1:.1%})' if v1 is not None else ''
        print(f'  {field:18s}  exact={a["exact"]:.1%}  (n={a.get("n")}){delta}')

fz = acc.get('brand', {}).get('fuzzy')
if fz is not None:
    print(f'  brand fuzzy:        {fz:.1%}')

p = metrics.get('vlm_price_eur', {})
if p.get('mape') is not None:
    print(f'  price MAPE:         {p["mape"]:.3f}   (vs GPT-4o-mini 1.025)')
    print(f'  price MAE:          €{p["mae"]:.2f}')


## Download the results

Run the cell below, **or** right-click each file in the Files panel → *Download*.


In [ ]:
from google.colab import files
files.download('eval/results/qwen_field_eval.json')
files.download('eval/results/qwen_field_predictions.parquet')


## (Optional) Back up to Google Drive


In [ ]:
import shutil
DEST = '/content/drive/MyDrive/Resell_Copilot_data'
shutil.copy('eval/results/qwen_field_eval.json', f'{DEST}/qwen_field_eval.json')
shutil.copy('eval/results/qwen_field_predictions.parquet', f'{DEST}/qwen_field_predictions.parquet')
print('backed up to', DEST)
